# Import a folder of protein structures

`projects.sync_proteins` adds every structure file in a folder to your project: `.pdb`, `.cif`,
`.mmcif` and `.ent`. It checks the files, uploads them, runs the import and waits for it to
finish, then returns every protein in the project as `Protein` objects, without downloading
their structures. You don't build a CSV or zip yourself.

In [ ]:
from deeporigin import projects
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient.from_disk()
projects.create("python-client-dev", client=client)

## Make a folder of structures

This example copies the BRD4 structure that ships with the SDK into a new folder. To import
your own structures, point `sync_proteins` at your folder instead.

In [ ]:
import shutil
import tempfile
from pathlib import Path

from deeporigin.drug_discovery import BRD_DATA_DIR

folder = Path(tempfile.mkdtemp()) / "structures"
folder.mkdir()
shutil.copy(BRD_DATA_DIR / "brd.pdb", folder / "brd4.pdb")
sorted(p.name for p in folder.iterdir())

## Import the folder

Each file becomes one protein, named after its file name without extension (`brd4.pdb`
becomes `brd4`). A structure that is already in the project is reused and keeps its existing
name. Pass `recursive=True` to include subfolders, or a list of files instead of a folder.

`sync_proteins` raises `DeepOriginException` if the import fails and `TimeoutError` if it
hasn't finished within `timeout` seconds (one hour by default).

In [ ]:
import pandas as pd

proteins = projects.sync_proteins(folder)
pd.DataFrame([{"name": p.name, "id": p.id, "pdb_id": p.pdb_id} for p in proteins])

## Import again

A structure that is already in the project is reused, so importing the same folder again adds
no proteins.

In [ ]:
again = projects.sync_proteins(folder)
if len(again) != len(proteins):
    raise RuntimeError(f"Expected {len(proteins)} proteins after re-import, found {len(again)}")
len(again)

## Load a structure

The returned proteins have no structure loaded. Call `download()` when you need the
coordinates.

In [ ]:
protein = next(p for p in proteins if p.remote_path)
protein.download()
protein

## Files are checked before upload

Every file is checked before anything is uploaded. If any file can't be imported (missing,
unsupported type, compressed such as `.pdb.gz`, empty, or with no atom records), one error
lists them all and nothing is uploaded.

In [ ]:
from deeporigin.exceptions import DeepOriginException

bad_folder = Path(tempfile.mkdtemp())
(bad_folder / "empty.pdb").write_text("")
shutil.copy(BRD_DATA_DIR / "brd.pdb", bad_folder / "brd4.pdb")
(bad_folder / "compressed.pdb.gz").write_bytes(b"not a structure")

try:
    projects.sync_proteins(bad_folder)
except DeepOriginException as error:
    print(error.title)
    print(error.body)